# SQL Business Analysis

## Business Objective

The objective of this notebook is to answer business questions using SQL on the cleaned e-commerce datasets.

This notebook demonstrates the ability to:

- Load cleaned CSV datasets into a relational database
- Query multiple related tables using SQL
- Use joins, aggregations, grouping, ordering, and date-based analysis
- Translate SQL outputs into business insights
- Prepare findings for dashboard development and executive reporting

The analysis is designed to simulate how a Data Analyst would answer business questions using SQL in a real e-commerce environment.

## SQL Analysis Questions

This notebook answers the following business questions:

1. What is the total revenue generated?
2. How many orders and customers are in the database?
3. What is the average order value?
4. Which product categories generate the most revenue?
5. Which products generate the most revenue?
6. Which brands generate the most revenue?
7. How does revenue trend by month?
8. Which cities generate the most revenue?
9. Who are the highest-value customers?
10. Which products have the highest review ratings?
11. What is the event funnel?
12. Which categories have strong revenue but weaker ratings?

Each question includes a SQL query, output, and business interpretation.

## Import Libraries

In [1]:
import pandas as pd
import getpass

from pathlib import Path
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus

## Set Project Directory

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
SQL_OUTPUT_DIR = PROJECT_ROOT / "sql"

SQL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis')

## Load Cleaned Datasets

The cleaned datasets produced during the Data Cleaning phase are loaded into pandas before being written into a MySQL database.

In [3]:
users = pd.read_csv(CLEAN_DATA_DIR / "users_clean.csv")
products = pd.read_csv(CLEAN_DATA_DIR / "products_clean.csv")
orders = pd.read_csv(CLEAN_DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(CLEAN_DATA_DIR / "order_items_clean.csv")
reviews = pd.read_csv(CLEAN_DATA_DIR / "reviews_clean.csv")
events = pd.read_csv(CLEAN_DATA_DIR / "events_clean.csv")

users["signup_date"] = pd.to_datetime(
    users["signup_date"],
    errors="coerce"
)

orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

reviews["review_date"] = pd.to_datetime(
    reviews["review_date"],
    errors="coerce"
)

events["event_timestamp"] = pd.to_datetime(
    events["event_timestamp"],
    errors="coerce"
)

## Create MySQL Database

A local MySQL database is created for SQL analysis. Each cleaned dataset is written as a database table.

In [4]:
MYSQL_USER = "root"
MYSQL_PASSWORD = getpass.getpass("Enter your MySQL password: ")
MYSQL_HOST = "localhost"
MYSQL_PORT = "3306"
MYSQL_DATABASE = "ecommerce_analytics"

encoded_password = quote_plus(MYSQL_PASSWORD)

server_engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{encoded_password}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}"
)

with server_engine.begin() as connection:
    connection.execute(
        text(
            f"CREATE DATABASE IF NOT EXISTS `{MYSQL_DATABASE}`"
        )
    )

server_engine.dispose()

engine = create_engine(
    f"mysql+mysqlconnector://{MYSQL_USER}:{encoded_password}"
    f"@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
)

datasets = {
    "users": users,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "reviews": reviews,
    "events": events
}

for table_name, dataframe in datasets.items():
    dataframe.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False
    )

print(f"Loaded {len(datasets)} tables into {MYSQL_DATABASE}.")

Enter your MySQL password:  ········


Loaded 6 tables into ecommerce_analytics.


## Helper Function

This function allows SQL queries to be executed and displayed as pandas DataFrames.

In [20]:
def run_query(query):
    with engine.connect() as connection:
        return pd.read_sql_query(
            text(query),
            connection
        )

# 1. Total Revenue

## Business Question

What is the total revenue generated by the business?

This is one of the most important executive KPIs.

In [6]:
query = """
SELECT
    ROUND(SUM(item_total), 2) AS total_revenue
FROM order_items;
"""

total_revenue = run_query(query)
total_revenue

,total_revenue
0,11918668.95


## Business Interpretation

Total revenue provides a top-level measure of business performance. This KPI will be used later in the dashboard as one of the main executive metrics.

# 2. Total Orders and Customers

## Business Question

How many orders and customers are represented in the cleaned database?

In [7]:
query = """
SELECT
    (SELECT COUNT(DISTINCT order_id) FROM orders) AS total_orders,
    (SELECT COUNT(DISTINCT user_id) FROM users) AS total_customers,
    (SELECT COUNT(DISTINCT product_id) FROM products) AS total_products;
"""

database_size = run_query(query)
database_size

,total_orders,total_customers,total_products
0,20000,10000,2000


## Business Interpretation

This query summarizes the scale of the database. Total orders, customers, and products provide context for all downstream business analysis.

# 3. Average Order Value

## Business Question

What is the average order value?

Average Order Value (AOV) helps measure how much customers spend per order on average.

In [8]:
query = """
SELECT
    ROUND(SUM(item_total) / COUNT(DISTINCT order_id), 2) AS average_order_value
FROM order_items;
"""

average_order_value = run_query(query)
average_order_value

,average_order_value
0,595.93


## Business Interpretation

Average Order Value is useful for evaluating customer purchasing behavior and can help guide pricing, bundling, and promotion strategies.

# 4. Revenue by Product Category

## Business Question

Which product categories generate the most revenue?

In [9]:
query = """
SELECT
    p.category,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold,
    COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

category_revenue = run_query(query)
category_revenue

,category,total_revenue,total_units_sold,total_orders
0,Electronics,4777773.14,5503.0,3745
1,Automotive,2432590.37,5615.0,3807
2,Home & Kitchen,1095696.59,5586.0,3790
3,Sports,929872.64,5399.0,3616
4,Clothing,707887.16,6189.0,4137
5,Beauty,532009.11,5887.0,3978
6,Toys,373113.25,6120.0,4114
7,Pet Supplies,344547.85,6295.0,4145
8,None,331004.63,1511.0,1088
9,Books,256089.94,5615.0,3757


## Business Interpretation

Category-level revenue analysis identifies the strongest business segments. High-revenue categories may deserve priority in marketing, inventory planning, and dashboard reporting.

# 5. Top 10 Products by Revenue

## Business Question

Which products generate the most revenue?

In [10]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category,
    p.brand
ORDER BY total_revenue DESC
LIMIT 10;
"""

top_products = run_query(query)
top_products

,product_id,product_name,category,brand,total_revenue,total_units_sold
0,P001527,Willow Result,Electronics,Willow,73275.12,33.0
1,P001024,Astra Pull,Electronics,Astra,65467.64,27.0
2,P000021,Willow Special,Electronics,Willow,62032.60,33.0
3,P000910,Orion Group,Electronics,Orion,60784.10,22.0
4,P000136,Zenith Phone,Electronics,Zenith,56419.48,24.0
5,P001925,Willow Hospital,Automotive,Willow,56311.92,41.0
6,P000461,Nimbus Family,Electronics,Nimbus,54411.20,32.0
7,P001049,Zenith Their,Electronics,Zenith,54399.32,34.0
8,P000198,Orion Face,Electronics,Orion,54105.20,39.0
9,P000555,Acme Of,Electronics,Acme,53489.18,37.0


## Business Interpretation

Top revenue-generating products are major business drivers. These products may be strong candidates for promotions, homepage placement, or inventory prioritization.

# 6. Revenue by Brand

## Business Question

Which brands generate the most revenue?

In [11]:
query = """
SELECT
    p.brand,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold,
    COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.brand
ORDER BY total_revenue DESC
LIMIT 10;
"""

brand_revenue = run_query(query)
brand_revenue

,brand,total_revenue,total_units_sold,total_orders
0,Willow,1333667.54,4994.0,3441
1,Orion,1156616.67,4874.0,3374
2,Nimbus,1091999.73,5267.0,3537
3,Acme,1043174.92,4606.0,3120
4,Astra,1023491.29,4830.0,3303
5,Zenith,1013086.63,5672.0,3871
6,GreenLeaf,989095.30,5232.0,3488
7,Harbor,921264.61,5174.0,3528
8,Solace,891005.52,4653.0,3187
9,Pulse,882109.91,4220.0,2875


## Business Interpretation

Brand-level revenue analysis can support supplier negotiations, brand partnerships, and merchandising strategy.

# 7. Monthly Revenue Trend

## Business Question

How does revenue change over time?

In [12]:
query = """
SELECT
    DATE_FORMAT(o.order_date, '%Y-%m') AS order_month,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders AS o
LEFT JOIN order_items AS oi
    ON o.order_id = oi.order_id
GROUP BY DATE_FORMAT(o.order_date, '%Y-%m')
ORDER BY order_month;
"""

monthly_revenue = run_query(query)
monthly_revenue

,order_month,total_revenue,total_orders
0,None,111986.26,197
1,2024-01,538454.34,876
2,2024-02,509380.40,806
3,2024-03,537379.40,941
4,2024-04,539664.42,923
5,2024-05,550227.28,897
6,2024-06,540183.89,886
7,2024-07,583351.79,952
8,2024-08,512987.69,869
9,2024-09,524981.05,881


## Business Interpretation

Monthly revenue trends help identify seasonality, growth patterns, or periods that require deeper investigation.

# 8. Revenue by City

## Business Question

Which cities generate the most revenue?

In [13]:
query = """
SELECT
    u.city,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT u.user_id) AS unique_customers
FROM orders o
LEFT JOIN users u
    ON o.user_id = u.user_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY u.city
ORDER BY total_revenue DESC
LIMIT 10;
"""

city_revenue = run_query(query)
city_revenue

,city,total_revenue,total_orders,unique_customers
0,None,413593.65,715,305
1,Nan,49906.01,86,36
2,North Michael,18549.63,23,9
3,Port James,15443.71,17,6
4,East James,14930.35,17,7
5,South Robert,14047.15,17,6
6,East Beth,13787.01,16,5
7,Lake Christopher,13526.77,19,7
8,Lake Alyssamouth,13288.88,4,1
9,North Williamville,12919.79,8,3


## Business Interpretation

City-level revenue analysis helps identify strong geographic markets. These insights can support regional marketing campaigns and logistics decisions.

# 9. Highest-Value Customers

## Business Question

Which customers generate the most revenue?

In [14]:
query = """
SELECT
    u.user_id,
    u.name,
    u.city,
    ROUND(SUM(oi.item_total), 2) AS customer_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM users u
LEFT JOIN orders o
    ON u.user_id = o.user_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    u.user_id,
    u.name,
    u.city
ORDER BY customer_revenue DESC
LIMIT 10;
"""

top_customers = run_query(query)
top_customers

,user_id,name,city,customer_revenue,total_orders
0,U009931,Meagan Case,Lake Alyssamouth,13288.88,4
1,U006233,Willie Esparza,Tylerland,11234.22,3
2,U006469,Audrey Ware,South Robertstad,10953.18,4
3,U008370,Brittany Carrillo,Bryanport,10190.11,2
4,U005702,Shawn Greene,New Ashleebury,10175.81,6
5,U009903,Jessica Russell,Nolanborough,10118.31,5
6,U000006,Michael Santiago,East Steven,9952.28,6
7,U004906,Brenda Winters,East Emilyview,9538.69,5
8,U006231,Jerome Warner,Port Katherineview,9486.29,2
9,U007930,Robert Jennings,Port James,8947.98,9


## Business Interpretation

High-value customers are important for retention strategy. These customers may be good candidates for loyalty programs, personalized offers, or VIP segmentation.

# 10. Highest Rated Products

## Business Question

Which products have the highest average review ratings?

In [15]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    ROUND(AVG(r.rating), 2) AS average_review_rating,
    COUNT(r.review_id) AS review_count
FROM reviews r
LEFT JOIN products p
    ON r.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
HAVING review_count >= 5
ORDER BY average_review_rating DESC, review_count DESC
LIMIT 10;
"""

highest_rated_products = run_query(query)
highest_rated_products

,product_id,product_name,category,average_review_rating,review_count
0,P001812,Zenith Partner,Automotive,4.67,9
1,P000863,Zenith Glass,None,4.67,7
2,P001288,Pulse Politics,Books,4.67,5
3,P001042,Pulse Degree,Home & Kitchen,4.60,5
4,P000694,Orion Even,Home & Kitchen,4.57,8
5,P001045,Pulse Front,Toys,4.57,7
6,P001990,Zenith Page,Home & Kitchen,4.50,8
7,P000618,Harbor Race,Clothing,4.50,8
8,P000447,Astra Shake,Beauty,4.50,6
9,P000522,Harbor Medical,Clothing,4.50,6


## Business Interpretation

Products with strong ratings and sufficient review volume may indicate high customer satisfaction. These products can be highlighted in marketing or used as benchmarks for quality.

# 11. Event Funnel

## Business Question

How are users moving through the behavioral funnel?

In [16]:
query = """
SELECT
    event_type,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_id) AS unique_users
FROM events
GROUP BY event_type
ORDER BY event_count DESC;
"""

event_funnel = run_query(query)
event_funnel

,event_type,event_count,unique_users
0,View,54389,9955
1,Cart,11682,6882
2,Wishlist,7708,5399
3,Purchase,3862,3176
4,None,2204,1972
5,Nan,155,154


## Business Interpretation

The event funnel shows how users interact with the platform. Comparing views, carts, wishlists, and purchases helps identify potential conversion opportunities or friction points.

# 12. Revenue and Rating by Category

## Business Question

Which categories generate strong revenue but may have weaker customer ratings?

In [17]:
query = """
SELECT
    p.category,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    ROUND(AVG(r.rating), 2) AS average_review_rating,
    COUNT(DISTINCT r.review_id) AS review_count
FROM products p
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
LEFT JOIN reviews r
    ON p.product_id = r.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

category_revenue_rating = run_query(query)
category_revenue_rating

,category,total_revenue,average_review_rating,review_count
0,Electronics,36337086.76,3.55,1422
1,Automotive,18571854.55,3.52,1365
2,Home & Kitchen,8242872.80,3.54,1410
3,Sports,7347465.02,3.52,1353
4,Clothing,5609254.02,3.55,1600
5,Beauty,4221772.76,3.57,1530
6,Toys,3062129.27,3.54,1601
7,Pet Supplies,2717133.37,3.58,1534
8,None,2636464.86,3.54,421
9,Books,2085902.78,3.54,1421


## Business Interpretation

This analysis compares commercial performance with customer satisfaction. Categories with high revenue but weaker ratings may require quality review, better product descriptions, or customer experience improvements.

# Export SQL Query Results

The main SQL outputs are exported as CSV files so they can be reused in reporting or dashboard development.

In [18]:
SQL_RESULTS_DIR = SQL_OUTPUT_DIR / "query_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

category_revenue.to_csv(SQL_RESULTS_DIR / "category_revenue.csv", index=False)
top_products.to_csv(SQL_RESULTS_DIR / "top_products.csv", index=False)
brand_revenue.to_csv(SQL_RESULTS_DIR / "brand_revenue.csv", index=False)
monthly_revenue.to_csv(SQL_RESULTS_DIR / "monthly_revenue.csv", index=False)
city_revenue.to_csv(SQL_RESULTS_DIR / "city_revenue.csv", index=False)
top_customers.to_csv(SQL_RESULTS_DIR / "top_customers.csv", index=False)
event_funnel.to_csv(SQL_RESULTS_DIR / "event_funnel.csv", index=False)

list(SQL_RESULTS_DIR.iterdir())

[PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/monthly_revenue.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/category_revenue.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/brand_revenue.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/top_customers.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/city_revenue.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/top_products.csv'),
 PosixPath('/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis/sql/query_results/event_funnel.csv')]

# Executive Summary

## Objective

The objective of this notebook was to answer key e-commerce business questions using SQL.

## SQL Skills Demonstrated

This notebook demonstrates:

- Aggregations using `SUM`, `COUNT`, and `AVG`
- Grouping with `GROUP BY`
- Sorting with `ORDER BY`
- Filtering grouped results with `HAVING`
- Joining multiple relational tables
- Date-based analysis using month extraction
- Exporting query results for reporting

## Key Business Areas Analyzed

The SQL analysis covered:

- Revenue performance
- Order and customer volume
- Average order value
- Category performance
- Product performance
- Brand performance
- Monthly revenue trends
- Geographic revenue
- High-value customers
- Product ratings
- Behavioral event funnel

## Business Value

The SQL analysis translates cleaned and integrated e-commerce data into business insights. These results can support decision-making in marketing, product strategy, inventory planning, customer retention, and dashboard development.

## Next Steps

The next stage of the project is Power BI dashboard development. The KPIs and SQL outputs from this notebook will guide dashboard structure, visual selection, and executive reporting priorities.

## Close Database Connection

In [19]:
engine.dispose()

print("MySQL connection closed.")

MySQL connection closed.
